In [1]:
#set things
#experiment_name = "test"
#config??
#modell?



#next:
#vibe code rewrite:
    #config manager
    #dataset splitter
    #couterfactual testing
    #prosessing hatespeech 100k
    #eval in generall
    #predicting
    #bootstraping



In [2]:
#for google colab 
#an kaggle
#clone repo and install requirements


from pathlib import Path
import subprocess
import os

IN_KAGGLE = (
    "KAGGLE_KERNEL_RUN_TYPE" in os.environ
    or Path("/kaggle/working").exists()
)

IN_COLAB = False

if not IN_KAGGLE:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        pass

if IN_COLAB:
    repo = Path("/content/gender-bias-hate-speech")

elif IN_KAGGLE:
    repo = Path("/kaggle/working/gender-bias-hate-speech")
    
else:
    # Settig root localy.
    repo = Path(r"C:\Users\lunaj\Desktop\gender-bias-hate-speech")


if not repo.exists():
        subprocess.run(
            ["git", "clone",
             "https://github.com/load-luna/gender-bias-hate-speech.git",
             str(repo)],
            check=True,
        )

os.chdir(repo)
results_dir = repo / "results"
results_dir.mkdir(exist_ok=True)

print(f"Project root: {Path.cwd()}")
print("Saving results to:", results_dir)


%pip install -r requirements.txt

# Workaround nur auf Colab
if IN_COLAB or IN_KAGGLE:
    %pip uninstall -y torchao

Project root: C:\Users\lunaj\Desktop\gender-bias-hate-speech
Saving results to: C:\Users\lunaj\Desktop\gender-bias-hate-speech\results
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# 2. IMPORTS
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

from transformers import set_seed
from peft import PeftConfig

from src.config import ExperimentConfig, prepare_run

from src.data.dataset import load_data_from_config, prepare_dataset, split_dataset
from src.data.text_processing import deep_clean_tweet
from src.data.gender_labeling import add_gender_label, count_label_distribution, swap_gender_terms_dataset, gender_labels

from src.data.splits import load_or_create_splits

from src.training.model import (
    load_model, load_tokenizer,
    tokenize_dataset,
    load_fine_tuned_model,
    prepare_lora,
    compute_metrics,
    check_max_sequence_length
)
from src.training.train import (
    create_training_arguments,
    create_trainer,
)

from src.training.predict import predict_dataset

from src.evaluation.metrics import eval_prediction
from src.evaluation.saving import save_predictions, save_counterfactual_predictions, save_evaluation



In [4]:
# 3. LOAD CONFIGURATION
# ============================================================

config = ExperimentConfig.from_files(
    model="configs/model.yaml",
    training="configs/training.yaml",
    evaluation="configs/evaluation.yaml",
    dataset="configs/datasets/3D_combined.yaml",
    mitigations=[
        "configs/mitigation/baseline.yaml",
    ],
)

#set variable about whether the model is fine-tuned or not
is_fine_tuned = config["model"].get("is_fine_tuned", False)

print("Resolved configuration:")
print(config.to_dict())


Resolved configuration:
{'model': {'name': 'bert-base-uncased', 'is_fine_tuned': False, 'path': 'google-bert/bert-base-uncased'}, 'lora': {'enabled': True, 'r': 8, 'alpha': 16, 'dropout': 0.1, 'target_modules': ['query', 'value'], 'bias': 'none', 'task_type': 'SEQ_CLS', 'modules_to_save': ['classifier']}, 'training': {'learning_rate': 5e-05, 'batch_size': 16, 'epochs': 8, 'weight_decay': 0.01, 'warmup_steps': 500, 'seed': 42, 'metric_for_best_model': 'f1_macro', 'greater_is_better': True, 'load_best_model_at_end': True}, 'evaluation': {'batch_size': 32, 'metrics': ['accuracy', 'f1', 'tpr_gap', 'equalized_odds'], 'counterfactual_testing': True}, 'dataset': {'name': '3D_combined', 'config_name': None, 'text_column': 'text', 'label_column': 'label', 'gender_column': 'gender_label', 'source': 'local', 'local': {'path': 'data/raw/3DatasetsCombined.csv'}, 'columns': {'text': 'text', 'label': 'class'}, 'label_mapping': {'mode': 'binary', 'num_labels': 2, 'mapping': {0: 0, 1: 1}, 'names': {0: 

In [5]:
# 4. CREATE RUN
# ============================================================



if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    results_root = Path(
        "/content/drive/MyDrive/gender-bias-hate-speech/results"
    )
else:
    results_root = repo / "results"

config, run_dir = prepare_run(
    experiment_name="baseline_3D_combined",
    config=config,
    results_root=results_root,
)

print(f"Run directory: {run_dir}")


Run directory: C:\Users\lunaj\Desktop\gender-bias-hate-speech\results\baseline_3D_combined\run_004


In [6]:
# 5. LOAD DATASET
# ============================================================

dataset = load_data_from_config(config)
dataset = prepare_dataset(dataset, config)

print(dataset)



Map:   0%|          | 0/48049 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'id', 'label', 'category'],
    num_rows: 48049
})


In [7]:
# 6. Clean dataset // text preprocessing
# ============================================================

dataset = dataset.map(deep_clean_tweet)


# Remove tweets reduced to empty text (e.g. URL-only tweets).
rows_before_filter = len(dataset)
dataset = dataset.filter(lambda example: bool(example["text"].strip()))
print(f"Removed {rows_before_filter - len(dataset)} empty texts after cleaning.")


Map:   0%|          | 0/48049 [00:00<?, ? examples/s]

Filter:   0%|          | 0/48049 [00:00<?, ? examples/s]

Removed 0 empty texts after cleaning.


In [8]:
# 7. add Gender Label
# ============================================================
dataset = dataset.map(add_gender_label)

print("Gender label distribution:")
count_label_distribution(dataset, config)

### add more analysis....


Map:   0%|          | 0/48049 [00:00<?, ? examples/s]

Gender label distribution:
           ['neither', 'hateful']
male       [2480, 2707]
female     [1395, 4206]
equal      [219, 556]
none       [16194, 20292]


[[2480, 2707], [1395, 4206], [219, 556], [16194, 20292]]

In [9]:

# Split dataset
dataset = load_or_create_splits(
    dataset, config, run_dir, copy_assignments=True,
)

In [10]:
# 7. Preprocessing MITIGATION
# ============================================================

mitigation_config = config.get(
    "mitigation",
    {}
)

print("\nMitigation configuration:")
print(mitigation_config)




Mitigation configuration:
{}


In [11]:
# 6. DATASET SANITY CHECKS
# ============================================================

print("\nDataset splits:")

for split in dataset.keys():
    print(f"{split}: {len(dataset[split])}")


print("\nColumns:")
print(dataset["train"].column_names)


print("\nFirst training example:")
print(dataset["train"][0])


# Target label distribution
if "label" in dataset["train"].column_names:

    print("\nLabel distribution:")

    print(
        dataset["train"]
        .to_pandas()["label"]
        .value_counts()
        .sort_index()
    )


# Gender-label distribution
if "gender_label" in dataset["train"].column_names:

    print("\nGender distribution:")

    print(
        dataset["train"]
        .to_pandas()["gender_label"]
        .value_counts()
        .sort_index()
    )




Dataset splits:
train: 28802
validation: 4804
test: 14443

Columns:
['text', 'id', 'label', 'category', 'gender_label']

First training example:
{'text': '"yes they do. our time is approaching."', 'id': 3007, 'label': 0, 'category': 'neither', 'gender_label': 3}

Label distribution:
label
0    12107
1    16695
Name: count, dtype: int64

Gender distribution:
gender_label
0     3061
1     3387
2      454
3    21900
Name: count, dtype: int64


In [12]:
#check fine-tuned model


if is_fine_tuned:
    model_path=config["model"]["path"]
    model_config = PeftConfig.from_pretrained(model_path)
    model_folder = Path(model_path)
    if not (model_folder / "adapter_config.json").is_file():
        raise FileNotFoundError(
            f"Keine adapter_config.json in {model_folder}. "
            "Bitte das gespeicherte Modell als Input hinzufügen und model_path anpassen."
        )
    if not any((model_folder / name).is_file() for name in
               ("adapter_model.safetensors", "adapter_model.bin")):
        raise FileNotFoundError(f"Keine LoRA-Gewichte in {model_folder} gefunden.")

    print(f"Model path: {model_path}")
    print(f"Model config: {model_config}")

In [13]:
# 8. LOAD TOKENIZER
# ============================================================

model_path = config["model"]["path"]

tokenizer_path = model_path

if is_fine_tuned and not (Path(model_path) / "tokenizer_config.json").exists():
    saved_config = PeftConfig.from_pretrained(model_path)
    tokenizer_path = saved_config.base_model_name_or_path
    if not tokenizer_path:
        raise ValueError("In adapter_config.json fehlt base_model_name_or_path.")

tokenizer = load_tokenizer(tokenizer_path)

print("\nTokenizer loaded:")
print(tokenizer.__class__.__name__)





Tokenizer loaded:
BertTokenizer


In [ ]:
# 9. TOKENIZE DATASET
# ============================================================

tokenized_dataset = tokenize_dataset(
    dataset=dataset,
    tokenizer=tokenizer,
    truncation=True
)

too_long = check_max_sequence_length(tokenizer, tokenized_dataset)

print(f"So many are textes are too long: {too_long}")

print("\nTokenized dataset:")
print(tokenized_dataset)

Map:   0%|          | 0/28802 [00:00<?, ? examples/s]

Map:   0%|          | 0/4804 [00:00<?, ? examples/s]

Map:   0%|          | 0/14443 [00:00<?, ? examples/s]

Filter:   0%|          | 0/28802 [00:00<?, ? examples/s]

Length of input_ids: 534
text: "japan does not have a welfare system. work for it or do without. and i don’t think there has ever been a better explanation of the importance of incentive than this example. an economics professor at a local college made a statement that he had never failed a single student before, but had recently failed an entire class. that class had insisted that socialism worked and that no one would be poor and no one would be rich, a great equalizer. the professor then said, "ok, we will have an experiment in this class on socialism". all grades will be averaged and everyone will receive the same grade so no one will fail and no one will receive an a. (substituting grades for dollars - something closer to home and more readily understood by all). after the first test, the grades were averaged and everyone got a b. the students who studied 'hard' were upset and the students who studied 'little' were happy. as the second test rolled around, the students who studied 

Filter:   0%|          | 0/4804 [00:00<?, ? examples/s]

Filter:   0%|          | 0/14443 [00:00<?, ? examples/s]

Length of input_ids: 639
text: "because in bills world he is always right and everyone else who disagrees is a pin head.or a zealot or a bigot. the man is a boar and bombastic. that sells you say. i will predict that his rein at the top is over, he inadvertently thought that his viewers were so sucked in that they would buy anything he says as gospel. he underestimated his audience badly. it pissed him off that the meek would dare to question his almighty wisdom. there are many people going to be raised up these days to try and prevent the takeover of this county from the rich establishment trying to tell the people what they should do. what we have found out is that most of fix have the same agenda as the dems,.to be in control! thats whats going on here. to control the money the masses and the airwaves. keep you distracted so you wont notice whats really going on. another sun tzu law. never underestimate your opponent, big mistake . tonight they are trying to remind the people their 

In [ ]:
# 10. LOAD MODEL
# ============================================================

# load_model() should:
#
# 1. load bert-base-uncased
# 2. create the sequence-classification head
# 3. apply LoRA
#
# LoRA is part of the standard model setup and therefore
# should happen inside src/training/model.py.


set_seed(config["training"]["seed"])

label_names = {
    int(key): value
    for key, value in config["dataset"]["label_mapping"]["names"].items()
}

#if fine-tuned model exists, load it. Otherwise, load the base model and prepare it for LoRA fine-tuning.
is_fine_tuned = config["model"].get("is_fine_tuned", False)

if is_fine_tuned:
    lora_model = load_fine_tuned_model(
        model_path=config["model"]["path"],
        label_names=label_names
    )
else:


    model = load_model(
        model_path=config["model"]["path"],
        label_names=label_names,
    )

    lora_model = prepare_lora(
        model=model,
        config=config
    )

print("\nModel loaded:")
print(lora_model.__class__.__name__)


# Check that LoRA is actually active.
if hasattr(lora_model, "print_trainable_parameters"):

    print("\nTrainable parameters:")

    lora_model.print_trainable_parameters()




In [ ]:
# 11. TRAINING ARGUMENTS
# ============================================================

training_args = create_training_arguments(
    config=config,
    output_dir = run_dir / "checkpoints",
)

print("\nTraining arguments:")
print(training_args)



In [ ]:
#temporary Weighted Loss Function

import torch
import torch.nn.functional as F

# Nur Labels aus dem Trainingsset verwenden
train_labels = torch.tensor(
    tokenized_dataset["train"]["label"],
    dtype=torch.long,
)

num_classes = config["dataset"]["label_mapping"]["num_labels"]
counts = torch.bincount(train_labels, minlength=num_classes)

if (counts == 0).any():
    raise ValueError("Im Trainingsset fehlt mindestens eine Klasse.")

class_weights = counts.sum() / (num_classes * counts.float())
print(class_weights)


def weighted_loss(outputs, labels, num_items_in_batch=None):
    logits = outputs.logits.float()
    weights = class_weights.to(logits.device)

    loss_sum = F.cross_entropy(
        logits,
        labels,
        weight=weights,
        reduction="sum",
    )

    denominator = (
        num_items_in_batch
        if num_items_in_batch is not None
        else labels.numel()
    )

    return loss_sum / denominator

In [ ]:
import torch

print("CUDA verfügbar:", torch.cuda.is_available())
print("Anzahl sichtbarer GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

training_args.fp16 = torch.cuda.is_available()



In [ ]:
# 12. CREATE TRAINER
# ============================================================


trainer = create_trainer(
    model=lora_model,
    tokenizer=tokenizer,
    training_args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics= compute_metrics,
    ##compute_loss_func = weighted_loss, ###Temp
)

print("\nTrainer created.")



In [ ]:
# 13. TRAIN MODEL
# ============================================================

if not is_fine_tuned:
    train_result = trainer.train()
    print("\nTraining finished.")

    print("\nTraining metrics:")
    print(train_result.metrics)



print("Trainer-Gerät:", trainer.args.device)
print("GPU-Anzahl pro Prozess:", trainer.args.n_gpu)
print("Anzahl Prozesse:", trainer.args.world_size)
print("FP16 aktiviert:", trainer.args.fp16)



In [ ]:
## save Modell


if not is_fine_tuned:
    print("Speichere Modell …")
    model_dir = run_dir / "model"
    trainer.save_model(str(model_dir))
    tokenizer.save_pretrained(str(model_dir))
else:
    print("ist schon gespeichert")

In [ ]:
## predictions on test set

test_predictions = predict_dataset(tokenized_dataset["test"],trainer)
#show label distribution on test set
count_label_distribution(dataset["test"], config)
# Save immediately, before plots or subsequent evaluation cells.
predictions_path = save_predictions(
    test_predictions, run_dir / "predictions.csv",
    label_names=trainer.model.config.id2label,
    metadata={"variant": "original", "model_path": str(config["model"]["path"]), "run_dir": str(run_dir)},
)


In [ ]:
# prdiction on swapped test set

#swap Gender Terms in Texts
test_dataset_swapped = dataset["test"].map(swap_gender_terms_dataset)

#add new Gender Lable
test_dataset_swapped = test_dataset_swapped.map(add_gender_label)

#check distribution after swap
count_label_distribution(test_dataset_swapped, config)


#tokenize
test_tokenize_dataset_swapped = tokenize_dataset(test_dataset_swapped,tokenizer)
#predict
test_prediction_swapped = predict_dataset(test_tokenize_dataset_swapped,trainer)


# Save immediately, before plots or subsequent evaluation cells.
predictions_path = save_predictions(
    test_prediction_swapped, 
    run_dir / "swapped_predictions.csv",
    label_names=trainer.model.config.id2label,
    metadata={"variant": "swapped", "model_path": str(config["model"]["path"]), "run_dir": str(run_dir)},
)


In [ ]:
save_counterfactual_predictions(
    test_predictions,
    test_prediction_swapped,
    run_dir / "paired_predictions.csv",

)

In [ ]:

#get label_names from config
label_names = config["dataset"]["label_mapping"]["names"]

names = {
    int(class_id): name
    for class_id, name in label_names.items()
}

class_ids = sorted(names)
labels = [names[class_id] for class_id in class_ids]





eval = eval_prediction(
    test_predictions,
    class_ids = class_ids, 
    label_names =labels,
    gender_labels = gender_labels
)

swapped_eval = eval_prediction(
    test_prediction_swapped,
    class_ids = class_ids, 
    label_names = labels,
    gender_labels = gender_labels
)

save_evaluation(
    eval,
    run_dir / "normal",
    #metadata={"variant": "original", "model_path": str(config["model"]["path"]), "run_dir": str(run_dir)},
)

save_evaluation(
    swapped_eval,
    run_dir / "swapped",
    #metadata={"variant": "swapped", "model_path": str(config["model"]["path"]), "run_dir": str(run_dir)},
)
